In [1]:
!pip install ultralytics rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.5 MB/s eta 0:00:00


In [2]:
import os
import cv2
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
import rasterio
from rasterio.windows import Window
from shapely.geometry import box
import albumentations as A
from ultralytics import YOLO
import torch
import geopandas as gpd

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Device: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU Device: Tesla T4


In [4]:
import os

ROOT = "/kaggle/input/datasets/hassanmojab/xview-dataset"

IMAGE_DIR = os.path.join(ROOT, 'train_images/train_images')
LABEL_PATH = os.path.join(ROOT, 'train_labels/xView_train.geojson')

OUTPUT_DIR = '/kaggle/working/xview_yolo'

os.makedirs(OUTPUT_DIR, exist_ok=True)

os.makedirs(f'{OUTPUT_DIR}/images/train', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/images/val', exist_ok=True)

os.makedirs(f'{OUTPUT_DIR}/labels/train', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/labels/val', exist_ok=True)

print("Image Dir:", IMAGE_DIR)
print("Label Path:", LABEL_PATH)

Image Dir: /kaggle/input/datasets/hassanmojab/xview-dataset/train_images/train_images
Label Path: /kaggle/input/datasets/hassanmojab/xview-dataset/train_labels/xView_train.geojson


In [5]:
print(os.path.exists(IMAGE_DIR))
print(os.path.exists(LABEL_PATH))

True
True


In [6]:
with open("/kaggle/input/datasets/hassanmojab/xview-dataset/train_labels/xView_train.geojson") as f:
    data = json.load(f)

In [7]:
file_path = "/kaggle/input/datasets/hassanmojab/xview-dataset/train_labels/xView_train.geojson"

# Read the file
gdf = gpd.read_file(file_path)


In [8]:
gdf.head()

,bounds_imcoords,edited_by,cat_id,type_id,ingest_time,index_right,image_id,point_geom,feature_id,grid_file,geometry
0,"2712,1145,2746,1177",wwoscarbecerril,1040010028371A00,73,2017-07-24 12:49:09.118000+00:00,2356,2355.tif,0101000020E6100000616E4E6406A256C03BE6ADA0D621...,374410,Grid2.shp,"POLYGON ((-90.5317 14.56604, -90.5317 14.56614..."
1,"2720,2233,2760,2288",wwoscarbecerril,1040010028371A00,73,2017-07-24 17:26:05.701000+00:00,2356,2355.tif,0101000020E6100000042D0CC705A256C0004F7071E71F...,394393,Grid2.shp,"POLYGON ((-90.53167 14.56222, -90.53167 14.562..."
2,"2687,1338,2740,1399",wwoscarbecerril,1040010028371A00,73,2017-07-24 12:45:09.081000+00:00,2356,2355.tif,0101000020E6100000B29F7E4707A256C0000AE6537921...,374031,Grid2.shp,"POLYGON ((-90.53179 14.56527, -90.53179 14.565..."
3,"2691,1201,2730,1268",wwoscarbecerril,1040010028371A00,73,2017-07-24 12:49:09.118000+00:00,2356,2355.tif,0101000020E6100000CE53137207A256C00EBF7084B521...,374409,Grid2.shp,"POLYGON ((-90.53177 14.56572, -90.53177 14.565..."
4,"2671,838,2714,869",wwoscarbecerril,1040010028371A00,73,2017-07-24 13:20:38.280000+00:00,2356,2355.tif,0101000020E610000060DE147608A256C05D6A384C6122...,377368,Grid2.shp,"POLYGON ((-90.53184 14.5671, -90.53184 14.5672..."


In [9]:
gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [10]:
gdf.bounds_imcoords

0         2712,1145,2746,1177
1         2720,2233,2760,2288
2         2687,1338,2740,1399
3         2691,1201,2730,1268
4           2671,838,2714,869
                 ...         
601932      710,1900,738,1925
601933      2091,258,2138,295
601934      2106,361,2124,379
601935        1917,38,1958,64
601936        2323,55,2349,74
Name: bounds_imcoords, Length: 601937, dtype: object

In [11]:
gdf.total_bounds

array([    -99.097,     -34.911,      151.23,      53.925])

In [12]:
# Returns: [min_lon, min_lat, max_lon, max_lat]
xmin, ymin, xmax, ymax = gdf.total_bounds
print(f"Bounding Box: Min X={xmin}, Min Y={ymin}, Max X={xmax}, Max Y={ymax}")


Bounding Box: Min X=-99.09673541783974, Min Y=-34.9111829655798, Max X=151.23403861073413, Max Y=53.92520692619612


In [13]:
individual_bounds = gdf.bounds
print(individual_bounds.head())

        minx       miny       maxx       maxy
0 -90.531699  14.566036 -90.531581  14.566145
1 -90.531672  14.562217 -90.531533  14.562408
2 -90.531785  14.565273 -90.531603  14.565484
3 -90.531772  14.565723 -90.531637  14.565953
4 -90.531841  14.567095 -90.531692  14.567202


In [14]:
# 1. Generate the bounds dataframe (returns columns: minx, miny, maxx, maxy)
df_bounds = gdf.bounds

# 2. Optional: Rename the columns to your exact desired format
df_bounds = df_bounds.rename(columns={
    'minx': 'xmin',
    'miny': 'ymin',
    'maxx': 'xmax',
    'maxy': 'ymax'
})

# print(df_bounds.head())
# df_bounds


## Manage the Coordinate of the box in image and pair them with image id and type id

In [15]:
with open(LABEL_PATH) as f:
    data = json.load(f)

features = data['features']

annotations = []

for feat in features:
    props = feat['properties']

    image_id = props['image_id']
    type_id = props['type_id']
    coords = props['bounds_imcoords']

    if coords == '':
        continue

    x1, y1, x2, y2 = map(int, coords.split(','))

    annotations.append([
        image_id,
        type_id,
        x1,
        y1,
        x2,
        y2
    ])

ann_df = pd.DataFrame(
    annotations,
    columns=['image_id', 'type_id', 'x1', 'y1', 'x2', 'y2']
)

ann_df.head()

,image_id,type_id,x1,y1,x2,y2
0,2355.tif,73,2712,1145,2746,1177
1,2355.tif,73,2720,2233,2760,2288
2,2355.tif,73,2687,1338,2740,1399
3,2355.tif,73,2691,1201,2730,1268
4,2355.tif,73,2671,838,2714,869


## Now perfrom the chipping method (Convert large image into small pieces), if image visibility is less than 40% then we will skip it.

In [16]:
CHIP_SIZE = 800 #640 #512    # 1024
STRIDE = 640 #512 #384    # 896
VISIBILITY_THRESHOLD = 0.4


def calculate_visibility(box_coords, chip_box):
    obj_box = box(*box_coords)
    
    # Check if the object itself has an area to avoid division by zero
    if obj_box.area <= 0:
        return 0
        
    intersection = obj_box.intersection(chip_box)
    if intersection.is_empty:
        return 0
        
    return intersection.area / obj_box.area

In [17]:
class_mapping = {
    k: idx for idx, k in enumerate(sorted(ann_df['type_id'].unique()))
}

print(class_mapping)

{np.int64(11): 0, np.int64(12): 1, np.int64(13): 2, np.int64(15): 3, np.int64(17): 4, np.int64(18): 5, np.int64(19): 6, np.int64(20): 7, np.int64(21): 8, np.int64(23): 9, np.int64(24): 10, np.int64(25): 11, np.int64(26): 12, np.int64(27): 13, np.int64(28): 14, np.int64(29): 15, np.int64(32): 16, np.int64(33): 17, np.int64(34): 18, np.int64(35): 19, np.int64(36): 20, np.int64(37): 21, np.int64(38): 22, np.int64(40): 23, np.int64(41): 24, np.int64(42): 25, np.int64(44): 26, np.int64(45): 27, np.int64(47): 28, np.int64(49): 29, np.int64(50): 30, np.int64(51): 31, np.int64(52): 32, np.int64(53): 33, np.int64(54): 34, np.int64(55): 35, np.int64(56): 36, np.int64(57): 37, np.int64(59): 38, np.int64(60): 39, np.int64(61): 40, np.int64(62): 41, np.int64(63): 42, np.int64(64): 43, np.int64(65): 44, np.int64(66): 45, np.int64(71): 46, np.int64(72): 47, np.int64(73): 48, np.int64(74): 49, np.int64(75): 50, np.int64(76): 51, np.int64(77): 52, np.int64(79): 53, np.int64(82): 54, np.int64(83): 55, n

In [18]:
import cv2

chip_count = 0
# Define the split (e.g., first 80% for train, rest for val)
unique_images = ann_df['image_id'].unique()
train_split = int(0.8 * len(unique_images))

for i, image_name in enumerate(tqdm(unique_images)):
    # Determine if this image goes to train or val folder
    split = 'train' if i < train_split else 'val'
    
    # Correct the path: check if TRAIN_DIR is defined, otherwise use IMAGE_DIR
    image_path = os.path.join(IMAGE_DIR, image_name)
    
    if not os.path.exists(image_path):
        continue

    image_annotations = ann_df[ann_df['image_id'] == image_name]

    with rasterio.open(image_path) as src:
        width = src.width
        height = src.height

        for y in range(0, height - CHIP_SIZE, STRIDE):
            for x in range(0, width - CHIP_SIZE, STRIDE):
                window = Window(x, y, CHIP_SIZE, CHIP_SIZE)
                chip = src.read(window=window)
                
                # Check if chip is empty or invalid
                if chip.shape[1] < CHIP_SIZE or chip.shape[2] < CHIP_SIZE:
                    continue
                    
                chip = np.transpose(chip, (1, 2, 0))
                chip_boxes = []
                chip_polygon = box(x, y, x + CHIP_SIZE, y + CHIP_SIZE)

                for _, row in image_annotations.iterrows():
                    visibility = calculate_visibility([row.x1, row.y1, row.x2, row.y2], chip_polygon)
                    
                    if visibility < VISIBILITY_THRESHOLD:
                        continue

                    # Local coordinates relative to the chip
                    nx1 = max(0, row.x1 - x)
                    ny1 = max(0, row.y1 - y)
                    nx2 = min(CHIP_SIZE, row.x2 - x)
                    ny2 = min(CHIP_SIZE, row.y2 - y)
                    
                    # Convert to YOLO format: class x_center y_center width height (normalized 0-1)
                    dw = 1.0 / CHIP_SIZE
                    dh = 1.0 / CHIP_SIZE
                    x_center = ((nx1 + nx2) / 2.0) * dw
                    y_center = ((ny1 + ny2) / 2.0) * dh
                    w = (nx2 - nx1) * dw
                    h = (ny2 - ny1) * dh
                    
                    chip_boxes.append([class_mapping[row.type_id], x_center, y_center, w, h])

                if len(chip_boxes) > 0:
                    chip_filename = f"chip_{chip_count}"
                    
                    # 1. Save the Image
                    img_out_path = os.path.join(OUTPUT_DIR, f'images/{split}', f"{chip_filename}.jpg")
                    cv2.imwrite(img_out_path, cv2.cvtColor(chip, cv2.COLOR_RGB2BGR))
                    
                    # 2. Save the Label (.txt file)
                    lbl_out_path = os.path.join(OUTPUT_DIR, f'labels/{split}', f"{chip_filename}.txt")
                    with open(lbl_out_path, 'w') as f:
                        for box_data in chip_boxes:
                            f.write(f"{box_data[0]} {box_data[1]:.6f} {box_data[2]:.6f} {box_data[3]:.6f} {box_data[4]:.6f}\n")
                    
                    chip_count += 1

print(f"Successfully created {chip_count} chips.")

100%|██████████| 847/847 [32:30<00:00,  2.30s/it]

Successfully created 8923 chips.


## Albumentations (For CLAHE, 90 Degree Rotation, Random Brightness)
* Improves low contrast satellite imagery
* Objects can face any direction in satellite images
* Simulates different weather conditions

In [19]:
transform = A.Compose([

    A.CLAHE(p=0.5),

    A.RandomRotate90(p=0.7),

    A.HorizontalFlip(p=0.5),

    A.VerticalFlip(p=0.5),

    A.RandomBrightnessContrast(p=0.4),

], bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels']
))

In [20]:
import os
import yaml

OUTPUT_DIR = '/kaggle/working/xview_yolo'

# Create required directories
dirs = [
    f'{OUTPUT_DIR}/images/train',
    f'{OUTPUT_DIR}/images/val',
    f'{OUTPUT_DIR}/labels/train',
    f'{OUTPUT_DIR}/labels/val'
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

# FULL xView class names list (Updated to include missing category indices up to 62 total)
xview_names = [
    'Fixed-wing Aircraft', 'Small Aircraft', 'Cargo Plane', 'Helicopter',
    'Passenger Vehicle', 'Small Car', 'Bus', 'Pickup Truck',
    'Utility Truck', 'Truck', 'Cargo Truck', 'Truck w/Box',
    'Truck Tractor', 'Trailer', 'Truck w/Flatbed',
    'Truck w/Liquid', 'Crane Truck', 'Railway Vehicle',
    'Passenger Car', 'Cargo Car', 'Flat Car', 'Tank car',
    'Locomotive', 'Maritime Vessel', 'Motorboat',
    'Sailboat', 'Tugboat', 'Barge', 'Fishing Vessel',
    'Ferry', 'Yacht', 'Container Ship', 'Oil Tanker',
    'Engineering Vehicle', 'Tower crane', 'Container Crane',
    'Reach Stacker', 'Straddle Carrier', 'Mobile Crane',
    'Dump Truck', 'Haul Truck', 'Scraper/Tractor',
    'Front loader/Bulldozer', 'Excavator', 'Cement Mixer',
    'Ground Grader', 'Hut/Tent', 'Shed', 'Building',
    'Aircraft Hangar', 'Damaged Building', 'Facility',
    'Construction Site', 'Vehicle Lot', 'Helipad',
    'Storage Tank', 'Shipping container lot',
    'Shipping Container', 'Pylon', 'Tower', 
    'Class 60 Placeholder', 'Class 61 Placeholder'  # Added to reach your dataset mapping cap of 62 classes
]

# YAML configuration dictionary 
data_yaml = {
    'path': OUTPUT_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 62,  # Fixed: Changed from 60 to 62 to match your mapping boundary
    'names': {i: name for i, name in enumerate(xview_names)}
}

# Save YAML configuration file safely
yaml_path = f'{OUTPUT_DIR}/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"data.yaml successfully verified and saved at: {yaml_path}")

data.yaml successfully verified and saved at: /kaggle/working/xview_yolo/data.yaml


In [21]:
import os

print(os.listdir('/kaggle/working/xview_yolo'))

['labels', 'data.yaml', 'images']


In [22]:
print(os.path.exists('/kaggle/working/xview_yolo/images/train'))
print(os.path.exists('/kaggle/working/xview_yolo/images/val'))
print(os.path.exists('/kaggle/working/xview_yolo/labels/train'))
print(os.path.exists('/kaggle/working/xview_yolo/labels/val'))


True
True
True
True


In [23]:
from ultralytics import YOLO

# Load the YOLOv8 medium model pre-trained weights
model = YOLO('yolov8m.pt')

# Train the model
model.train(
    data = f'{OUTPUT_DIR}/data.yaml',
    epochs = 50,
    imgsz = 800,        # Update: Changed from 640 to match your CHIP_SIZE
    batch = 8,          # Update: Lowered from 16 to prevent Tesla T4 memory crash
    cache = False,      # Update: Set to False to protect Kaggle system RAM
    workers = 2,        # Tweak: Slightly reduced to lower CPU overhead
    mosaic = 1.0, 
    copy_paste = 0.3, 
    degrees = 90, 
    hsv_h = 0.015, 
    hsv_s = 0.7, 
    hsv_v = 0.4, 
    scale = 0.5, 
    translate = 0.1, 
    project = 'xview_yolo', 
    name = 'satellite_detector',
)

Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/xview_yolo/data.yaml, degrees=90, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=satellite_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=T

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ae7e588a960>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003, 

In [24]:
## from ultralytics import YOLO

# # Load the YOLOv8 medium model pre-trained weights
# model = YOLO('yolov8m.pt')

# # Train the model 
# model.train(
#     data=f'{OUTPUT_DIR}/data.yaml',
#     epochs=50,
#     imgsz=640,
#     batch=16,
#     # Let Ultralytics automatically select the active GPU/CPU accelerator
#     # instead of hardcoding device=0 to avoid device image issues
#     mosaic=1.0,
#     flipud=0.5,  # Enhanced: Up-down flips
#     fliplr=0.5,  # Enhanced: Left-right flips
#     mixup=0.1,   # Enhanced: Mixup for generalization
#     optimizer='auto', # Enhanced: Auto optimizer handles class weights better
#     copy_paste=0.3,
#     degrees=180, # Enhanced: Random rotations for satellite objects

#     hsv_h=0.015,
#     hsv_s=0.7,
#     hsv_v=0.4,

#     scale=0.5,
#     translate=0.1,

#     workers=4,
#     cache=True,

#     project='xview_yolo',
#     name='satellite_detector'
# )

In [25]:
metrics = model.val()

print(metrics.box.map50)
print(metrics.box.map)

Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,875,658 parameters, 0 gradients, 78.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2420.8±1020.9 MB/s, size: 191.8 KB)
val: Scanning /kaggle/working/xview_yolo/labels/val.cache... 1926 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1926/1926 807.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 121/121 1.4it/s 1:27
                   all       1926     148178      0.327      0.255      0.209      0.109
   Fixed-wing Aircraft         25         47       0.63     0.0638      0.319      0.141
        Small Aircraft         25         40      0.604      0.775      0.651      0.279
           Cargo Plane         57        129      0.676      0.953      0.905      0.569
            Helicopter          4          4     0.0851       0.25      0.256      0.176
     Passenger Vehicle        1

In [26]:
model.export(format='engine')

WARNING ⚠️ TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/kaggle/working/runs/detect/xview_yolo/satellite_detector/weights/best.pt' with input shape (1, 3, 800, 800) BCHW and output shape(s) (1, 66, 13125) (49.7 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 360ms
 Downloaded onnxruntime-gpu
Prepared 2 packages in 3.13s
Installed 2 packages in 12ms
 + onnxruntime-gpu==1.26.0
 + onnxslim==0.1.93

requirements: AutoUpdate success ✅ 4.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 18...
ONNX: slimming wit

PosixPath('/kaggle/working/runs/detect/xview_yolo/satellite_detector/weights/best.engine')

In [27]:
results = model.predict(
    source=f'{OUTPUT_DIR}/images/train/chip_0.jpg',
    conf=0.25,
    save=True
)


image 1/1 /kaggle/working/xview_yolo/images/train/chip_0.jpg: 800x800 3 Truck w/Boxs, 8 Pylons, 55.5ms
Speed: 2.9ms preprocess, 55.5ms inference, 1.7ms postprocess per image at shape (1, 3, 800, 800)
Results saved to /kaggle/working/runs/detect/predict


### 🌟 Enhancement: Large Image Inference & Stitching
This section implements the ability to run YOLOv8 on massive `.tif` satellite images by dividing them into manageable tiles, running inference, and mapping the predictions back to the original global coordinates.

In [ ]:
import rasterio
from rasterio.windows import Window
import numpy as np
import cv2
import torch
import torchvision
from ultralytics import YOLO

def predict_large_image(image_path, model_path, tile_size=800, overlap=100, iou_threshold=0.45):
    """
    Runs YOLOv8 inference on a large satellite image by tiling it and applies NMS.
    
    Args:
        image_path (str): Path to the large .tif image.
        model_path (str): Path to the trained YOLO weights (e.g. best.pt).
        tile_size (int): Size of the tile for inference.
        overlap (int): Overlap between tiles to prevent cutting objects.
        iou_threshold (float): Intersection Over Union threshold for NMS.
        
    Returns:
        list: Global bounding boxes [x1, y1, x2, y2, confidence, class_id] after NMS.
    """
    print(f"Loading model {model_path}...")
    model = YOLO(model_path)
    stride = tile_size - overlap
    all_boxes = []
    all_scores = []
    all_classes = []
    
    print(f"Processing large image: {image_path}")
    with rasterio.open(image_path) as src:
        width, height = src.width, src.height
        
        for y in range(0, height, stride):
            for x in range(0, width, stride):
                w = min(tile_size, width - x)
                h = min(tile_size, height - y)
                
                window = Window(x, y, w, h)
                img = src.read([1, 2, 3], window=window)
                img = np.transpose(img, (1, 2, 0)) # CHW to HWC
                
                # Avoid passing tiny fragments
                if img.shape[0] < 10 or img.shape[1] < 10:
                    continue
                
                # Run inference on the tile
                results = model.predict(img, imgsz=tile_size, verbose=False)
                
                for r in results:
                    for box in r.boxes:
                        b = box.xyxy[0].cpu().numpy()
                        conf = box.conf[0].cpu().numpy()
                        cls = box.cls[0].cpu().numpy()
                        
                        # Map local tile coordinates to global image coordinates
                        gx1, gy1 = b[0] + x, b[1] + y
                        gx2, gy2 = b[2] + x, b[3] + y
                        
                        all_boxes.append([gx1, gy1, gx2, gy2])
                        all_scores.append(conf)
                        all_classes.append(cls)
                        
    if not all_boxes:
        print("No objects detected.")
        return []

    # --- Apply Global Non-Maximum Suppression (NMS) ---
    boxes_tensor = torch.tensor(all_boxes, dtype=torch.float32)
    scores_tensor = torch.tensor(all_scores, dtype=torch.float32)
    classes_tensor = torch.tensor(all_classes, dtype=torch.float32)
    
    # batched_nms applies NMS separately for each class
    keep_indices = torchvision.ops.batched_nms(
        boxes=boxes_tensor,
        scores=scores_tensor,
        idxs=classes_tensor,
        iou_threshold=iou_threshold
    )
    
    final_predictions = []
    for idx in keep_indices:
        i = idx.item()
        final_predictions.append([
            all_boxes[i][0], all_boxes[i][1], all_boxes[i][2], all_boxes[i][3],
            all_scores[i].item(), all_classes[i].item()
        ])
        
    print(f"Completed inference. Found {len(all_boxes)} total objects, reduced to {len(final_predictions)} after NMS.")
    return final_predictions

# ==========================================
# Example Usage:
# ==========================================
# sample_image = f'{IMAGE_DIR}/2355.tif'
# trained_model = f'{OUTPUT_DIR}/satellite_detector/weights/best.pt'
# 
# if os.path.exists(sample_image) and os.path.exists(trained_model):
#     global_boxes = predict_large_image(sample_image, trained_model, iou_threshold=0.45)
#     print(f"First 5 predictions: {global_boxes[:5]}")
# else:
#     print("Train the model first to generate weights for inference!")
